# 17 · Long-term memory with the store

**Goal:** by the end you can save information that every thread can read, such as each user's preferred number of
decimal places, using a store, and explain how that differs from a checkpointer.

## The idea in plain English

The checkpointer from lesson 13 remembers **one thread**. Start a new thread, and it starts from nothing. That's
right for a single conversation, but some things should be remembered **across all of a user's threads**: "Alice
likes 4 decimal places" should still be true in her conversation tomorrow.

A **store** is long-term memory shared by every thread. It holds small dicts called **items**. Each item lives under
a **namespace** (a tuple of strings, like a folder path, here `("users", "alice")`) and a **key** (its name in that
folder, here `"prefs"`). You `put` items, `get` them back by namespace and key, and `search` a namespace to list what's
in it.

You give a graph a store with `compile(store=...)`. Nodes reach it through `runtime.store`, the same `Runtime` object
that carries the context in lesson 12.

**Analogy:** a doctor's office. The notes from today's visit are the checkpointer: they belong to this visit (thread).
Your patient file in the cabinet is the store: it's the same file at every visit, filed under your name (namespace).

## Picture

Three threads, one store. The checkpointer keeps each thread separate; the store is shared by all of them.

```mermaid
flowchart LR
    subgraph checkpointer["checkpointer: one history per thread"]
        t1["alice-monday"]
        t2["alice-tuesday"]
        t3["bob-monday"]
    end
    store[("store<br/>('users','alice') / prefs: 4 places<br/>('users','bob') / (nothing yet)")]
    t1 -- put --> store
    t2 -- get --> store
    t3 -- get --> store
```

## Part 1: The store on its own

### Step 1: `put` and `get`

**`InMemoryStore`** keeps items in memory (like `InMemorySaver`, it's lost when the kernel stops). `put(namespace,
key, value)` saves a dict. `get(namespace, key)` returns an **`Item`**: its `value`, plus the namespace, key and
timestamps.

In [1]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
store.put(("users", "alice"), "prefs", {"decimal_places": 4})

item = store.get(("users", "alice"), "prefs")
print("value:    ", item.value)
print("namespace:", item.namespace)
print("key:      ", item.key)
print("created:  ", item.created_at)

value:     {'decimal_places': 4}
namespace: ('users', 'alice')
key:       prefs
created:   2026-10-04 08:00:45.823215+00:00


### Step 2: Missing items return `None`

Asking for something that was never saved isn't an error. You get `None`, so always check before reading `.value`.

In [2]:
print(store.get(("users", "bob"), "prefs"))

None


### Step 3: `search` a namespace

`search(namespace)` returns every item in that namespace, **and** in any namespace that starts with it. So
`("users",)` finds items for every user.

In [3]:
store.put(("users", "alice"), "nickname", {"name": "Al"})
store.put(("users", "carol"), "prefs", {"decimal_places": 0})

print("alice's items:")
for item in store.search(("users", "alice")):
    print("  ", item.key, item.value)

print("every user's items:")
for item in store.search(("users",)):
    print("  ", item.namespace, item.key, item.value)

alice's items:
   prefs {'decimal_places': 4}
   nickname {'name': 'Al'}
every user's items:
   ('users', 'alice') prefs {'decimal_places': 4}
   ('users', 'alice') nickname {'name': 'Al'}
   ('users', 'carol') prefs {'decimal_places': 0}


## Part 2: A calculator that remembers preferences

### Step 4: Context, state and the nodes

The user's ID is a run-level setting, so it goes in the **context** (lesson 12). A caller may pass `decimal_places`
in the input to **set** a preference. If they don't, the saved preference is used.

- `remember_preference` saves `decimal_places` to the store when it was given.
- `calculate` works out the result, as usual.
- `format_answer` looks the preference up in the store. If there is none, it uses 2.
- `log_calculation` saves every answer to a second namespace, `("users", id, "calculations")`, using a new random key
  each time (`uuid4`), so all of a user's calculations can be listed later.

In [4]:
from dataclasses import dataclass
from typing import NotRequired, TypedDict


@dataclass
class Context:
    user_id: str


class CalcState(TypedDict):
    expression: str
    decimal_places: NotRequired[int]  # optional: pass it to set a preference
    result: float
    answer: str

`NotRequired[int]` marks a key that may be missing from the state. It's only a hint, like everything in a `TypedDict`
(lesson 02), but it tells the reader "you don't have to pass this".

In [5]:
from uuid import uuid4

from langgraph.runtime import Runtime


def remember_preference(state: CalcState, runtime: Runtime[Context]) -> dict:
    if "decimal_places" in state:
        namespace = ("users", runtime.context.user_id)
        runtime.store.put(namespace, "prefs", {"decimal_places": state["decimal_places"]})
        print(f"  saved preference for {runtime.context.user_id}: {state['decimal_places']} places")
    return {}

`calculate` does the arithmetic, as in lesson 12. It doesn't use the store.

In [6]:
def calculate(state: CalcState) -> dict:
    left, op, right = state["expression"].split()
    a, b = float(left), float(right)
    if op == "+":
        return {"result": a + b}
    elif op == "-":
        return {"result": a - b}
    elif op == "*":
        return {"result": a * b}
    return {"result": a / b}

`format_answer` looks up the user's preference (falling back to 2), and `log_calculation` saves each answer under a new random key.

In [7]:
def format_answer(state: CalcState, runtime: Runtime[Context]) -> dict:
    prefs = runtime.store.get(("users", runtime.context.user_id), "prefs")
    places = prefs.value["decimal_places"] if prefs else 2  # default when nothing is saved
    return {"answer": f"{state['expression']} = {state['result']:.{places}f}"}


def log_calculation(state: CalcState, runtime: Runtime[Context]) -> dict:
    namespace = ("users", runtime.context.user_id, "calculations")
    runtime.store.put(namespace, str(uuid4()), {"answer": state["answer"]})
    return {}

### Step 5: Compile with a checkpointer **and** a store

They do different jobs, so a graph often has both. We use a fresh store so the earlier test items don't get in the
way.

In [8]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END

builder = StateGraph(CalcState, context_schema=Context)
builder.add_node("remember_preference", remember_preference)
builder.add_node("calculate", calculate)
builder.add_node("format", format_answer)
builder.add_node("log", log_calculation)
builder.add_edge(START, "remember_preference")
builder.add_edge("remember_preference", "calculate")
builder.add_edge("calculate", "format")
builder.add_edge("format", "log")
builder.add_edge("log", END)

store = InMemoryStore()
graph = builder.compile(checkpointer=InMemorySaver(), store=store)
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	remember_preference(remember_preference)
	calculate(calculate)
	format(format)
	log(log)
	__end__([<p>__end__</p>]):::last
	__start__ --> remember_preference;
	calculate --> format;
	format --> log;
	remember_preference --> calculate;
	log --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### Step 6: Alice sets a preference on Monday

A small helper, `run`, saves typing: it builds the config for a thread and passes the user in the context.

In [9]:
def run(thread_id: str, user_id: str, inputs: dict) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    return graph.invoke(inputs, config, context={"user_id": user_id})["answer"]


print(run("alice-monday", "alice", {"expression": "10 / 3", "decimal_places": 4}))

  saved preference for alice: 4 places
10 / 3 = 3.3333


### Step 7: A new thread on Tuesday still knows

`alice-tuesday` is a **new thread**, so the checkpointer has nothing for it. But the store is shared across threads,
so Alice's 4 decimal places are still used.

In [10]:
print(run("alice-tuesday", "alice", {"expression": "22 / 7"}))

22 / 7 = 3.1429


Bob has never set a preference, so he gets the default of 2 places.

In [11]:
print(run("bob-monday", "bob", {"expression": "22 / 7"}))

22 / 7 = 3.14


### Step 8: Checkpointer vs store, side by side

The checkpointer's view of `alice-tuesday` holds only Tuesday's calculation, not Monday's. The store holds what's true
for Alice in every thread, including a log of **all** her calculations.

In [12]:
tuesday = graph.get_state({"configurable": {"thread_id": "alice-tuesday"}})
print("checkpointer, alice-tuesday:", tuesday.values)
print()
print("store, alice's prefs:       ", store.get(("users", "alice"), "prefs").value)
print("store, alice's calculations:")
for item in store.search(("users", "alice", "calculations")):
    print("  ", item.value["answer"])

checkpointer, alice-tuesday: {'expression': '22 / 7', 'result': 3.142857142857143, 'answer': '22 / 7 = 3.1429'}

store, alice's prefs:        {'decimal_places': 4}
store, alice's calculations:
   10 / 3 = 3.3333
   22 / 7 = 3.1429


| | Checkpointer (lesson 13) | Store (this lesson) |
|---|---|---|
| Remembers | The state of **one thread** | Items shared by **every thread** |
| Organised by | `thread_id` and checkpoint | Namespace + key |
| Written | Automatically, after every step | Only when a node calls `put` |
| Read | Automatically, at the start of a run | When a node calls `get` or `search` |
| Good for | One conversation's state | User preferences, facts learned over time |
| Given to the graph by | `compile(checkpointer=...)` | `compile(store=...)` |

Like `InMemorySaver`, `InMemoryStore` is lost when the kernel stops. For memory on disk, the installed
`langgraph-checkpoint-sqlite` package also provides `SqliteStore` (in `langgraph.store.sqlite`).

## What just happened

- **Step 1** put `{'decimal_places': 4}` under `('users', 'alice')` / `prefs` and got it back as an `Item`.
- **Step 2** showed a missing item returns `None`.
- **Step 3** searched `('users', 'alice')` (two items) and `('users',)` (every user's items).
- **Step 6** ran Alice's Monday thread with `decimal_places: 4`. The node saved the preference, and the answer was
  `10 / 3 = 3.3333`.
- **Step 7** started a brand-new thread, `alice-tuesday`. It still used 4 places, `22 / 7 = 3.1429`, because it read
  the store. Bob got the default, `22 / 7 = 3.14`.
- **Step 8** showed the checkpointer's Tuesday thread knows only Tuesday's calculation, while the store had Alice's
  preference and a log of both her calculations.

## Common mistakes

1. **Forgetting `compile(store=...)`.** Then `runtime.store` is `None`, and the first `get` fails:
   `AttributeError: 'NoneType' object has no attribute 'get'`.
   Fix: `builder.compile(store=InMemoryStore())`.
2. **Reading `.value` of a missing item.** `store.get(...)` returns `None` when nothing was saved, so
   `store.get(...).value` fails:
   `AttributeError: 'NoneType' object has no attribute 'value'`.
   Fix: check first, as in `prefs.value[...] if prefs else default`.
3. **Using the store for one conversation's working data.** There's **no error**, but values from one thread leak
   into every other thread of that user.
   Fix: keep a conversation's state in the graph state (with a checkpointer). Put only what should be shared in the
   store.

## Try it

1. Write a function `forget_preferences(user_id)` that deletes the user's `prefs` item with
   `store.delete(namespace, key)`. Delete Alice's preference, then run a new thread for her. Which number of decimal
   places is used now?
2. Carol sets `decimal_places: 0` in a thread called `carol-1`. In another thread, `carol-2`, she calculates
   `"7 / 2"`. What answer does she get? Then print how many calculations the store has logged for Carol.

Answers are in [`solutions/exercises_solutions.ipynb`](../solutions/exercises_solutions.ipynb).

## Key terms

New terms from this lesson, also in [GLOSSARY.md](../GLOSSARY.md):

- **Store**: long-term memory shared by every thread. Given to a graph with `compile(store=...)`.
- **`InMemoryStore`**: a store kept in memory. Lost when the program stops.
- **Namespace**: a tuple of strings, like a folder path, that groups items, such as `("users", "alice")`.
- **Item**: one saved dict in the store, with its `namespace`, `key`, `value` and timestamps.
- **`put` / `get` / `search` / `delete`**: save, read one, list a namespace, remove one.
- **`NotRequired`**: a `TypedDict` hint marking a key that may be left out.